# Estimate Mode Choice 2

Estimate mode choice models of TNC vs transit/walk

Uses combined HH travel survey + TNC data.  Updates availability from version 1, focusing on weighted estimation. 

In [22]:
import numpy as np

import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.reset_option('display.float_format')

import biogeme.biogeme as bio
import biogeme.database as biodb
from biogeme import models
from biogeme.expressions import Beta, Variable

In [23]:
# read the data
df = pd.read_csv('out/combined_estimation_file.csv')
df.head()

C:\Users\ger225\AppData\Local\Temp\ipykernel_25796\360866534.py:2: DtypeWarning: Columns (0: depart_date, 1: linked_trip_mode_labeled, 2: income_labeled) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('out/combined_estimation_file.csv')


,Unnamed: 0,hh_id,person_id,person_num,day_id,day_num,depart_date,o_tract_2020,d_tract_2020,linked_trip_id,linked_trip_num,linked_trip_mode,linked_trip_weight,linked_trip_mode_labeled,mode,mode2,distance_miles,duration_minutes,o_district,d_district,o_community,d_community,time_period,ff_car_time_minutes,car_ivt,tnc_wait,tnc_time,tnc_fare,transit_fare,walk_time,transit_time,transit_or_walk_time,walk_faster_than_transit,transit_or_walk_fare,income_broad,income_labeled,hh_share_inc_under_100k,hh_share_inc_over_100k,tnc_trip_id,obs_fare,obs_tip,obs_additional_charges,transit_avail,walk_avail,transit_or_walk_avail,tnc_time_2,tnc_fare_2,time_period_num,avg_obs_tnc_time,avg_obs_tnc_fare,tnc_observed,tnc_time_3,tnc_fare_3,tnc_time_minus_transit_walk,tnc_cost_minus_transit_walk
0,0,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031081500,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.810270,20.0,Downtown,Downtown,32.0,8.0,midday,3.738333,6.186942,5,11.186942,6.707731,2.5,16.205401,22.0,16.205401,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,11.186942,6.707731,3,NaN,NaN,False,11.186942,6.707731,-5.018459,6.707731
1,1,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081500,17031081403,2.400012e+15,2.0,15.0,1853.792592,Walk,walk,walk,0.338027,28.0,Downtown,Downtown,8.0,8.0,midday,1.660000,2.747300,5,7.747300,4.798943,2.5,6.760535,7.0,6.760535,True,0.0,5.0,"$150,000 or more",0.321678,0.678322,NaN,NaN,NaN,NaN,1,1,1,7.747300,4.798943,3,7.157143,9.642857,True,12.157143,9.642857,5.396607,9.642857
2,2,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081403,17031320101,2.400012e+15,3.0,15.0,1853.792592,Walk,walk,walk,0.549293,15.0,Downtown,Downtown,8.0,32.0,midday,3.421667,5.662858,5,10.662858,6.244886,2.5,10.985870,23.0,10.985870,True,0.0,5.0,"$150,000 or more",0.460539,0.539461,NaN,NaN,NaN,NaN,1,1,1,10.662858,6.244886,3,NaN,NaN,False,10.662858,6.244886,-0.323012,6.244886
3,3,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320101,2.400012e+15,4.0,15.0,1853.792592,Walk,walk,walk,0.319386,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.167457,2.5,6.387712,12.0,6.387712,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,8.643758,5.167457,3,NaN,NaN,False,8.643758,5.167457,2.256047,5.167457
4,4,24000124.0,2.400012e+09,2.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320102,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.751861,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.561010,2.5,15.037220,12.0,12.000000,False,2.5,5.0,"$150,000 or more",0.341000,0.659000,NaN,NaN,NaN,NaN,1,1,1,8.643758,5.561010,3,NaN,NaN,False,8.643758,5.561010,-3.356242,3.061010


In [24]:
# which columns have NaNs, and how many
df.isna().sum()


Unnamed: 0                          0
hh_id                          156150
person_id                      156150
person_num                     156150
day_id                         156150
day_num                        156150
depart_date                    156150
o_tract_2020                        0
d_tract_2020                        0
linked_trip_id                 156150
linked_trip_num                156150
linked_trip_mode               156150
linked_trip_weight                  0
linked_trip_mode_labeled       156150
mode                                0
mode2                               0
distance_miles                      0
duration_minutes                    0
o_district                          0
d_district                          0
o_community                         0
d_community                         0
time_period                         0
ff_car_time_minutes                 0
car_ivt                             0
tnc_wait                            0
tnc_time    

In [25]:
# drop recrods where we don't know the income shares

trips_before = len(df)

df = df[df['hh_share_inc_under_100k']>0]
df = df[df['hh_share_inc_under_100k']>0]

print("Before: " + str(trips_before) + " After: " + str(len(df)))

Before: 161249 After: 161105


In [26]:
# fill the remaining missing values with zeros to make biogeme happy--BE CAREFUL!
df = df.fillna(0)

In [27]:
# add a flag for trips made by people in HHs with <$100k, $100k+ and missing annual income
# income_broad: 
# 1	Under $30,000
# 2	$30,000-$59,999
# 3	$60,000-$99,999
# 4	$100,000-$149,999
# 5	$150,000 or more
# 999	Prefer not to answer

df['inc_under_100k'] = np.where(df['income_broad']<=3, 1, 0)
df['inc_over_100k']  = np.where((df['income_broad']==4) | (df['income_broad']==5), 1, 0)
df['inc_missing']    = np.where((df['income_broad']==999), 1, 0)

In [28]:
# update availability

# transit must be less than 2 hours
df['transit_avail'] = np.where(df['transit_time']<120, 1, 0)

# walk must be less than 30 minutes
df['walk_avail'] = np.where(df['walk_time']<30, 1, 0)

# drop trips that choose an unavailable alternative
trips_before = len(df)
df = df[(df['transit_avail']==1) | (df['mode']!='transit')].copy()
df = df[(df['walk_avail']==1) | (df['mode']!='walk')].copy()

print("Before: " + str(trips_before) + " After: " + str(len(df)))

Before: 161105 After: 160922


In [29]:
# calculate normalized weights
df['normalized_weights'] = df['linked_trip_weight'] / df['linked_trip_weight'].sum() * len(df)

In [30]:
# Biogeme needs a NUMERIC choice column: tnc=1, transit=2, walk=3 and only numeric values in its database format. 
df['CHOICE'] = df['mode'].map({'tnc': 1, 'transit': 2, 'walk': 3})
df['BINARY_CHOICE'] = df['mode'].map({'tnc': 1, 'transit': 2, 'walk': 2})

df_numeric = df.select_dtypes(include='number').copy()

db = biodb.Database('mode_choice', df_numeric)

# Try weighted estimation

Normally I would use an unweighted estimation.  Here I try a weighted estimation since I think it will affect primarily the ASCs.  What we see below is that it is that the time and cost coefficients are smaller in magnitude, but the ASCs aren't much different.  I think we're better off sticking with the unweighted estimation, which is the norm. 

In [31]:
# Normally I would use an unweighted estimation, but here I care about the ASCs, so I will try weighting it.

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST      = Beta('B_COST',      0, None, None, 0)   # generic, shared across modes

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')
CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST * tnc_fare
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST * transit_fare
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST']['Value']
print("\nValue of Time: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		4
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64273.48
Likelihood ratio test (null):		109233.6
Rho square (null):			0.459
Rho bar square (null):			0.459
Akaike Information Criterion:	128555
Bayesian Information Criterion:	128594.9

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  2.046454      0.024656    82.999426           0.0
ASC_WALK     3.861205      0.027584   139.981201           0.0
B_COST      -0.040030      0.001966   -20.361972           0.0
B_TIME      -0.025225      0.000648   -38.953992           0.0

Value of Time: 37.81


# Consider zonal incomes instead of HH level incomes

In the TNC data, we won't actually observe the HH level incomes due to privacy restrictions.  Instead try segmenting VOT based on the income distribution in the Census tract of the trip's origin. 



In [32]:
# weighted estimation with zonal incomes
# Normally I would use an unweighted estimation, but here I care about the ASCs, so I will try weighting it.

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k 
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64270.27
Likelihood ratio test (null):		109240
Rho square (null):			0.459
Rho bar square (null):			0.459
Akaike Information Criterion:	128550.5
Bayesian Information Criterion:	128600.5

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  2.051704      0.024670    83.166871           0.0
ASC_WALK     3.869623      0.027581   140.300805           0.0
B_COST_HI   -0.033535      0.003139   -10.682862           0.0
B_COST_LOW  -0.043980      0.002615   -16.817358           0.0
B_TIME      -0.025167      0.000651   -38.661620           0.0

Value of Time for HH <$100k: 34.33
Value of Time for HH $100k+: 45.03


That's getting closer.  The ACSs seem high, as are the VOTs, but they are at least in the right order. 

In [33]:
# weighted estimation with zonal incomes
# Exclue the transit fare, which is probably discounted for most transit riders. 

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time 
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64264.38
Likelihood ratio test (null):		109251.8
Rho square (null):			0.459
Rho bar square (null):			0.459
Akaike Information Criterion:	128538.8
Bayesian Information Criterion:	128588.7

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  1.958783      0.028350    69.091725           0.0
ASC_WALK     3.873931      0.027537   140.678376           0.0
B_COST_HI   -0.030272      0.002888   -10.483118           0.0
B_COST_LOW  -0.045780      0.002491   -18.375987           0.0
B_TIME      -0.025134      0.000655   -38.386145           0.0

Value of Time for HH <$100k: 32.94
Value of Time for HH $100k+: 49.82


That's a better VOT distribution, although still higher than I would expect. 

In [34]:
# weighted estimation with zonal incomes
# Exclue the transit fare, which is probably discounted for most transit riders. 
# Add constant segmented by income

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
ASC_TRANSIT_HI = Beta('ASC_TRANSIT_HI', 0, None, None, 0) # constant on share of high-income HHs
ASC_WALK_HI    = Beta('ASC_WALK_HI',    0, None, None, 0) # constant on share of high-income HHs  
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time + ASC_TRANSIT_HI * hh_share_inc_over_100k  
V_walk    = ASC_WALK    + B_TIME * walk_time    + ASC_WALK_HI * hh_share_inc_over_100k  

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		7
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64229.9
Likelihood ratio test (null):		109320.8
Rho square (null):			0.46
Rho bar square (null):			0.46
Akaike Information Criterion:	128473.8
Bayesian Information Criterion:	128543.7

                   Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT     2.162048      0.076759    28.166644  0.000000e+00
ASC_TRANSIT_HI -0.432539      0.149428    -2.894623  3.796140e-03
ASC_WALK        3.870522      0.077079    50.214688  0.000000e+00
ASC_WALK_HI    -0.007620      0.150090    -0.050767  9.595114e-01
B_COST_HI      -0.042902      0.005272    -8.137037  4.440892e-16
B_COST_LOW     -0.035135      0.004751    -7.394956  1.414424e-13
B_TIME         -0.025184      0.000668   -37.708705  0.000000e+00

Value of Time for HH <$100k: 43.01
Value of Time for HH $100k+: 35.22


That is worse than unsegmented.

What if I use the observed TNC time and fare when they are available? 

In [35]:
# weighted estimation with zonal incomes
# Exclue the transit fare, which is probably discounted for most transit riders. 
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time_2')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_2')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time  
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64345.6
Likelihood ratio test (null):		109089.3
Rho square (null):			0.459
Rho bar square (null):			0.459
Akaike Information Criterion:	128701.2
Bayesian Information Criterion:	128751.1

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  1.706014      0.027561    61.900015           0.0
ASC_WALK     3.672737      0.026586   138.144567           0.0
B_COST_HI   -0.028692      0.003225    -8.896805           0.0
B_COST_LOW  -0.065812      0.002631   -25.013559           0.0
B_TIME      -0.020032      0.000683   -29.329451           0.0

Value of Time for HH <$100k: 18.26
Value of Time for HH $100k+: 41.89


We're getting closer.  I think there are two changes that might help:

1. Test a smarter way of allocating the income. 
2. Be smarter about using more realistic times and costs for all options.  

# Use probabilistically assigned income based on the share in each Census tract. 


In [36]:
rng = np.random.default_rng(42)
df['random'] = rng.random(len(df))

df['imputed_income_under_100k'] = np.where(df['random']<df['hh_share_inc_under_100k'], 1, 0)
df['imputed_income_over_100k'] = 1 - df['imputed_income_under_100k']

df.head()

,Unnamed: 0,hh_id,person_id,person_num,day_id,day_num,depart_date,o_tract_2020,d_tract_2020,linked_trip_id,linked_trip_num,linked_trip_mode,linked_trip_weight,linked_trip_mode_labeled,mode,mode2,distance_miles,duration_minutes,o_district,d_district,o_community,d_community,time_period,ff_car_time_minutes,car_ivt,tnc_wait,tnc_time,tnc_fare,transit_fare,walk_time,transit_time,transit_or_walk_time,walk_faster_than_transit,transit_or_walk_fare,income_broad,income_labeled,hh_share_inc_under_100k,hh_share_inc_over_100k,tnc_trip_id,obs_fare,obs_tip,obs_additional_charges,transit_avail,walk_avail,transit_or_walk_avail,tnc_time_2,tnc_fare_2,time_period_num,avg_obs_tnc_time,avg_obs_tnc_fare,tnc_observed,tnc_time_3,tnc_fare_3,tnc_time_minus_transit_walk,tnc_cost_minus_transit_walk,inc_under_100k,inc_over_100k,inc_missing,normalized_weights,CHOICE,BINARY_CHOICE,random,imputed_income_under_100k,imputed_income_over_100k
0,0,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031081500,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.810270,20.0,Downtown,Downtown,32.0,8.0,midday,3.738333,6.186942,5,11.186942,6.707731,2.5,16.205401,22.0,16.205401,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,0,0.0,0.0,0.0,1,1,1,11.186942,6.707731,3,0.000000,0.000000,False,11.186942,6.707731,-5.018459,6.707731,0,1,0,111.651873,3,2,0.773956,0,1
1,1,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081500,17031081403,2.400012e+15,2.0,15.0,1853.792592,Walk,walk,walk,0.338027,28.0,Downtown,Downtown,8.0,8.0,midday,1.660000,2.747300,5,7.747300,4.798943,2.5,6.760535,7.0,6.760535,True,0.0,5.0,"$150,000 or more",0.321678,0.678322,0,0.0,0.0,0.0,1,1,1,7.747300,4.798943,3,7.157143,9.642857,True,12.157143,9.642857,5.396607,9.642857,0,1,0,111.651873,3,2,0.438878,0,1
2,2,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031081403,17031320101,2.400012e+15,3.0,15.0,1853.792592,Walk,walk,walk,0.549293,15.0,Downtown,Downtown,8.0,32.0,midday,3.421667,5.662858,5,10.662858,6.244886,2.5,10.985870,23.0,10.985870,True,0.0,5.0,"$150,000 or more",0.460539,0.539461,0,0.0,0.0,0.0,1,1,1,10.662858,6.244886,3,0.000000,0.000000,False,10.662858,6.244886,-0.323012,6.244886,0,1,0,111.651873,3,2,0.858598,0,1
3,3,24000124.0,2.400012e+09,1.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320101,2.400012e+15,4.0,15.0,1853.792592,Walk,walk,walk,0.319386,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.167457,2.5,6.387712,12.0,6.387712,True,0.0,5.0,"$150,000 or more",0.341000,0.659000,0,0.0,0.0,0.0,1,1,1,8.643758,5.167457,3,0.000000,0.000000,False,8.643758,5.167457,2.256047,5.167457,0,1,0,111.651873,3,2,0.697368,0,1
4,4,24000124.0,2.400012e+09,2.0,2.400012e+11,1.0,2024-05-21,17031320101,17031320102,2.400012e+15,1.0,15.0,1853.792592,Walk,walk,walk,0.751861,16.0,Downtown,Downtown,32.0,32.0,midday,2.201667,3.643758,5,8.643758,5.561010,2.5,15.037220,12.0,12.000000,False,2.5,5.0,"$150,000 or more",0.341000,0.659000,0,0.0,0.0,0.0,1,1,1,8.643758,5.561010,3,0.000000,0.000000,False,8.643758,5.561010,-3.356242,3.061010,0,1,0,111.651873,3,2,0.094177,1,0


In [37]:
# rebuild the biogeme data

df_numeric = df.select_dtypes(include='number').copy()

db = biodb.Database('mode_choice', df_numeric)

In [38]:
# weighted estimation with zonal incomes
# Exclue the transit fare, which is probably discounted for most transit riders. 
# Add constant segmented by income
# use observed TNC time/fare where available
# use imputed individual income instead of zonal income

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time_2')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_2')
transit_fare = Variable('transit_fare')

imputed_income_under_100k = Variable('imputed_income_under_100k')
imputed_income_over_100k = Variable('imputed_income_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * imputed_income_under_100k + B_COST_HI * tnc_fare * imputed_income_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time  
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64380.73
Likelihood ratio test (null):		109019.1
Rho square (null):			0.458
Rho bar square (null):			0.458
Akaike Information Criterion:	128771.5
Bayesian Information Criterion:	128821.4

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  1.677937      0.027621    60.749647           0.0
ASC_WALK     3.642253      0.026798   135.914917           0.0
B_COST_HI   -0.056367      0.002589   -21.774133           0.0
B_COST_LOW  -0.048409      0.002159   -22.422595           0.0
B_TIME      -0.020517      0.000681   -30.127137           0.0

Value of Time for HH <$100k: 25.43
Value of Time for HH $100k+: 21.84


Interesting.  That made the incomes basically indistinguishable.  

In [39]:
# weighted estimation with zonal incomes
# Add constant segmented by income
# use imputed individual income instead of zonal income
# try an earlier specification with the original time and cost.

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')

imputed_income_under_100k = Variable('imputed_income_under_100k')
imputed_income_over_100k = Variable('imputed_income_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * imputed_income_under_100k + B_COST_HI * tnc_fare * imputed_income_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * imputed_income_under_100k + B_COST_HI * transit_fare * imputed_income_over_100k  
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64267.88
Likelihood ratio test (null):		109244.8
Rho square (null):			0.459
Rho bar square (null):			0.459
Akaike Information Criterion:	128545.8
Bayesian Information Criterion:	128595.7

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  2.043499      0.024667    82.844643           0.0
ASC_WALK     3.855295      0.027656   139.399565           0.0
B_COST_HI   -0.043723      0.002331   -18.755873           0.0
B_COST_LOW  -0.038006      0.002029   -18.727741           0.0
B_TIME      -0.025414      0.000650   -39.085684           0.0

Value of Time for HH <$100k: 40.12
Value of Time for HH $100k+: 34.88


# Back to initial income specification.

I like my original approach better.  I'm not sure this random assignment makes sense for estimation.  I think we have to tolerate some inconsistency between estimation and application. 

In [40]:
# this is our current preferred model

# weighted estimation with zonal incomes
# Exclue the transit fare, which is probably discounted for most transit riders. 
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time_2')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_2')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time  
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64345.6
Likelihood ratio test (null):		109089.3
Rho square (null):			0.459
Rho bar square (null):			0.459
Akaike Information Criterion:	128701.2
Bayesian Information Criterion:	128751.1

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  1.706014      0.027561    61.900009           0.0
ASC_WALK     3.672737      0.026586   138.144555           0.0
B_COST_HI   -0.028692      0.003225    -8.896811           0.0
B_COST_LOW  -0.065812      0.002631   -25.013558           0.0
B_TIME      -0.020032      0.000683   -29.329453           0.0

Value of Time for HH <$100k: 18.26
Value of Time for HH $100k+: 41.89


In [20]:
# if we do include the transit fare

# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time_2')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_2')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k 
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			161118
Excluded data:			0
Null log likelihood:		-118995.3
Final log likelihood:		-64459.26
Likelihood ratio test (null):		109072.2
Rho square (null):			0.458
Rho bar square (null):			0.458
Akaike Information Criterion:	128928.5
Bayesian Information Criterion:	128978.5

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  1.830976      0.023921    76.543369  0.000000e+00
ASC_WALK     3.680529      0.027030   136.162094  0.000000e+00
B_COST_HI   -0.027464      0.003504    -7.836922  4.662937e-15
B_COST_LOW  -0.065664      0.002826   -23.236919  0.000000e+00
B_TIME      -0.019934      0.000682   -29.214085  0.000000e+00

Value of Time for HH <$100k: 18.21
Value of Time for HH $100k+: 43.55


# Update time and cost

This is close, but we can be smarter about the TNC time and cost for the transit and walk alternatives.  Use observed values for TNC trips between the same zone pairs where they are available. 

In [41]:
# add our best estimate of TNC time and fare

# include the transit fare
# weighted estimation with zonal incomes
# Add constant segmented by income
# use observed TNC time/fare where available

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)    
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time_3')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare_3')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k = Variable('hh_share_inc_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k 
V_walk    = ASC_WALK    + B_TIME * walk_time    

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}         

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			160922
Excluded data:			0
Null log likelihood:		-118890.3
Final log likelihood:		-64088.56
Likelihood ratio test (null):		109603.4
Rho square (null):			0.461
Rho bar square (null):			0.461
Akaike Information Criterion:	128187.1
Bayesian Information Criterion:	128237.1

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  1.706218      0.025159    67.818318           0.0
ASC_WALK     3.520884      0.028358   124.156244           0.0
B_COST_HI   -0.039199      0.003356   -11.680922           0.0
B_COST_LOW  -0.077109      0.003006   -25.651117           0.0
B_TIME      -0.021315      0.000657   -32.428708           0.0

Value of Time for HH <$100k: 16.59
Value of Time for HH $100k+: 32.63


# Woohoo!  We have a winner!